# ERCOT LSTM hyperparameter tuning

Screen 11 settings on 2024?2025 validation data, confirm the top two candidates and the original on three seeds, and score the frozen choice on shared 2026 hours. Run the cells in order. Completed trials are reused from `optimization/results/lstm/checkpoints/`.

## 1. Set up the search

The grid varies lookback, hidden units, and learning rate. Settings and input hashes allow completed trials to be reused with the same environment.

In [1]:
import copy
import hashlib
import json
import platform
import time
from pathlib import Path

from IPython.display import display

import numpy as np
import pandas as pd
import torch
from pandas.tseries.holiday import USFederalHolidayCalendar
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


working_dir = Path.cwd().resolve()
locations = (
    working_dir,
    working_dir / "optimization",
    working_dir / "final project" / "ERCOT Project" / "optimization",
)
HERE = next(
    (path for path in locations if path.name == "optimization" and (path.parent / "preliminary").is_dir()),
    None,
)
if HERE is None:
    raise FileNotFoundError("Start the kernel from optimization/, the ERCOT Project root, or the course workspace.")
PRELIMINARY = HERE.parent / "preliminary"
NOTEBOOK = PRELIMINARY / "ercot_lstm_forecast.ipynb"
DATA = PRELIMINARY / "ercot_system_hourly_cleaned.csv"
BASELINE = PRELIMINARY / "lstm_outputs" / "lstm_config.json"
OUTPUT = HERE / "results" / "lstm"
SEEDS = (42, 7, 2026)
TRIALS = (
    (7, 64, 0.03), (28, 64, 0.03), (14, 32, 0.03), (14, 128, 0.03),
    (14, 64, 0.01), (14, 64, 0.05), (7, 32, 0.05), (7, 128, 0.01),
    (28, 32, 0.01), (28, 128, 0.05),
)
TUNABLES = ("lookback", "hidden", "lr")



def write_json(path, value):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


CPU_THREADS = 4
torch.set_num_threads(CPU_THREADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
original = json.loads(BASELINE.read_text(encoding="utf-8"))["config"]
configs = {"original": original}
configs.update({f"candidate_{i:02}": {**original, **dict(zip(TUNABLES, trial))}
                for i, trial in enumerate(TRIALS, 1)})
checkpoint_dir = OUTPUT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
settings = {
    "configs": configs, "seeds": list(SEEDS), "device": str(device), "threads": CPU_THREADS,
    "python": platform.python_version(), "processor": platform.processor(),
    "torch": str(torch.__version__), "numpy": np.__version__, "pandas": pd.__version__,
    "inputs": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (NOTEBOOK, DATA, BASELINE)},
}
path = OUTPUT / "settings.json"
if path.exists() and json.loads(path.read_text(encoding="utf-8")) != settings:
    raise ValueError("Saved trials have different settings. Archive the result directory before starting a new run.")
write_json(path, settings)


## 2. Prepare training and validation

Use data and training definitions from the preliminary LSTM notebook. The tuning tensors only cover years through 2025. Each completed trial saves its validation forecasts, selected weights, and metrics.

In [2]:
def notebook_pipeline(config, device, include_test=False):
    """Load only the data and model definitions used by the notebook's tuning hook."""
    cells = ["".join(cell["source"]) for cell in json.loads(NOTEBOOK.read_text(encoding="utf-8"))["cells"]]
    raw = pd.read_csv(DATA, parse_dates=["timestamp"])
    if not include_test:
        raw = raw.loc[raw["timestamp"].dt.year <= 2025].copy()
    namespace = {
        "raw": raw, "CONFIG": config, "device": device, "HORIZON": 24, "SEED": SEEDS[0],
        "np": np, "pd": pd, "torch": torch, "nn": nn, "copy": copy, "time": time,
        "USFederalHolidayCalendar": USFederalHolidayCalendar,
        "DataLoader": DataLoader, "TensorDataset": TensorDataset,
    }
    for start, stop in (
        ("import time, json", None), ("NUM_COLS =", None), ("# Day x hour matrices", None),
        ("def cyc(", "\nDATA = make_data(CONFIG)"), ("class LSTMForecaster", None),
    ):
        cell = next(source for source in cells if source.startswith(start))
        if start == "import time, json":
            cell = "def set_seed(" + cell.split("\ndef set_seed(", 1)[1]
        elif stop is not None:
            cell, marker, _ = cell.partition(stop)
            if not marker:
                raise ValueError(f"Notebook cell is missing its expected boundary {stop!r}.")
        exec(compile(cell, str(NOTEBOOK), "exec"), namespace)
    return namespace


def sample_rows(data, split):
    return data["S"]["rows"][data["masks"][split]]


def metrics(actual, predicted):
    error = predicted - actual
    return {"mae_mw": float(np.abs(error).mean()), "rmse_mw": float(np.sqrt((error ** 2).mean()))}


def run_trial(pipeline, config, data, seed, checkpoint):
    if checkpoint.exists():
        result = torch.load(checkpoint, map_location="cpu", weights_only=True)
    else:
        model, history = pipeline["train_model"](config, data, seed=seed)
        val = pipeline["forecast_mw"](model, data, "val")
        rows = sample_rows(data, "val")
        if not np.isfinite(val).all():
            raise ValueError(f"Non-finite validation predictions for {checkpoint.stem}.")
        # Match the notebook's checkpoint rule, which requires improvement by 1e-5.
        best_loss, best_epoch = float("inf"), 0
        for epoch, loss in zip(history["epoch"], history["val_loss"]):
            if loss < best_loss - 1e-5:
                best_loss, best_epoch = loss, epoch
        result = {
            "validation": torch.from_numpy(val),
            "metrics": {
                **metrics(pipeline["L"][rows], val), "best_epoch": int(best_epoch),
                "epochs_run": len(history), "train_seconds": float(history["time"].iloc[-1]),
            },
            "weights": {key: param.detach().cpu() for key, param in model.state_dict().items()},
        }
        temporary = checkpoint.with_suffix(".tmp")
        torch.save(result, temporary)
        temporary.replace(checkpoint)
    print(f"{checkpoint.stem}: validation MAE {result['metrics']['mae_mw']:.2f} MW", flush=True)
    return result



pipeline = notebook_pipeline(original, device)
data_by_window = {}
checkpoints, trial_records = {}, {}


def run(name, seed):
    cfg = configs[name]
    window = cfg["lookback"]
    if window not in data_by_window:
        data_by_window[window] = pipeline["make_data"](cfg)
    data = data_by_window[window]
    result = run_trial(pipeline, cfg, data, seed, checkpoint_dir / f"{name}_seed{seed}.pt")
    checkpoints[name, seed] = result
    trial_records[name, seed] = {"name": name, "seed": seed, **{key: cfg[key] for key in TUNABLES}, **result["metrics"]}
    pd.DataFrame(trial_records.values()).to_csv(OUTPUT / "validation_trials.csv", index=False)


Interpolated single hours: 8
Hours filled from previous week: 0
Grid: 2018-01-01 00:00:00 -> 2025-12-31 23:00:00 (2922 days)
train    2191
val       731
Name: count, dtype: int64 
Uri days: 7 | unusable target days: []


## 3. Screen 11 settings with seed 42

Rank all settings by validation MAE in MW. Carry the two best candidates and the original into the three-seed comparison.

In [3]:
for name in configs:
    run(name, SEEDS[0])
ranked = pd.DataFrame(trial_records[name, SEEDS[0]] for name in configs).sort_values("mae_mw")
ranked.to_csv(OUTPUT / "screening.csv", index=False)
finalists = ["original", *ranked.loc[ranked["name"] != "original", "name"].iloc[:2]]


display(ranked.reset_index(drop=True))


original_seed42: validation MAE 1645.63 MW
candidate_01_seed42: validation MAE 1624.78 MW
candidate_02_seed42: validation MAE 1639.50 MW
candidate_03_seed42: validation MAE 1650.56 MW
candidate_04_seed42: validation MAE 1660.87 MW
candidate_05_seed42: validation MAE 1765.92 MW
candidate_06_seed42: validation MAE 1639.89 MW
candidate_07_seed42: validation MAE 1633.48 MW
candidate_08_seed42: validation MAE 1673.26 MW
candidate_09_seed42: validation MAE 1662.89 MW
candidate_10_seed42: validation MAE 1657.34 MW


,name,seed,lookback,hidden,lr,mae_mw,rmse_mw,best_epoch,epochs_run,train_seconds
0,candidate_01,42,7,64,0.03,1624.776092,2505.128259,95,120,25.396073
1,candidate_07,42,7,32,0.05,1633.476489,2485.709554,70,95,16.272824
2,candidate_02,42,28,64,0.03,1639.497356,2503.761975,111,136,56.710088
3,candidate_06,42,14,64,0.05,1639.894492,2495.429035,52,77,21.723372
4,original,42,14,64,0.03,1645.630515,2493.098041,58,83,24.450034
5,candidate_03,42,14,32,0.03,1650.562217,2491.121536,98,123,26.725759
6,candidate_10,42,28,128,0.05,1657.342121,2517.137177,101,126,105.254913
7,candidate_04,42,14,128,0.03,1660.867391,2496.182238,74,99,51.257097
8,candidate_09,42,28,32,0.01,1662.890796,2526.186964,135,160,44.245738
9,candidate_08,42,7,128,0.01,1673.255976,2534.585648,135,160,52.382365


## 4. Confirm and freeze the selection

Average predictions across seeds 42, 7, and 2026 before scoring each ensemble. Freeze the lowest-validation-MAE setting before running the test cell.

In [4]:
ensemble = []
for name in finalists:
    for seed in SEEDS[1:]:
        run(name, seed)
    cfg = configs[name]
    rows = sample_rows(data_by_window[cfg["lookback"]], "val")
    prediction = np.mean([checkpoints[name, seed]["validation"].numpy() for seed in SEEDS], axis=0)
    ensemble.append({"name": name, **{key: cfg[key] for key in TUNABLES},
                     **metrics(pipeline["L"][rows], prediction)})
ensemble = pd.DataFrame(ensemble).sort_values("mae_mw", kind="stable")
ensemble.to_csv(OUTPUT / "validation_ensembles.csv", index=False)
chosen = ensemble.iloc[0]["name"]
write_json(OUTPUT / "selection.json", {"name": chosen, "config": configs[chosen], "seeds": SEEDS})
print(f"Frozen selection: {chosen}")
display(ensemble.reset_index(drop=True))


original_seed7: validation MAE 1622.42 MW
original_seed2026: validation MAE 1640.39 MW
candidate_01_seed7: validation MAE 1622.44 MW
candidate_01_seed2026: validation MAE 1661.69 MW
candidate_07_seed7: validation MAE 1651.70 MW
candidate_07_seed2026: validation MAE 1679.03 MW
Frozen selection: original


,name,lookback,hidden,lr,mae_mw,rmse_mw
0,original,14,64,0.03,1621.871647,2475.195829
1,candidate_01,7,64,0.03,1622.954327,2480.021672
2,candidate_07,7,32,0.05,1636.348944,2485.159393


## 5. Evaluate the frozen choice on shared 2026 hours

Score the saved original, its reproduction, and a tuned model if one was selected on validation. Verify matching timestamps and observed loads on the 301 shared days.

In [5]:
def test_results(original, configs, chosen, checkpoints, device):
    pipeline = notebook_pipeline(original, device, include_test=True)
    ml = pd.read_csv(PRELIMINARY / "ercot_ml_24hour_forecasts.csv", parse_dates=["timestamp"])
    saved = pd.read_csv(PRELIMINARY / "lstm_outputs" / "lstm_day_ahead_forecasts.csv", parse_dates=["timestamp"])
    saved = saved.loc[saved["split"] == "test", ["timestamp", "actual_mw", "lstm_mw"]]
    shared = ml.merge(saved, on="timestamp", validate="one_to_one").sort_values("timestamp")
    counts = shared.groupby(shared["timestamp"].dt.normalize()).size()
    if len(counts) != 301 or not (counts == 24).all():
        raise ValueError("Expected 301 complete shared forecast days.")
    actual = shared["actual_mw"].to_numpy()
    np.testing.assert_array_equal(actual, shared["actual_load_mw"].to_numpy())
    forecasts = shared[["timestamp", "actual_mw"]].copy()
    forecasts["saved_original"] = shared["lstm_mw"]
    labels = {"original": "original_rerun"}
    if chosen != "original":
        labels[chosen] = "tuned"

    for name, label in labels.items():
        cfg = configs[name]
        data = pipeline["make_data"](cfg)
        rows = sample_rows(data, "test")
        model = pipeline["LSTMForecaster"](
            data["n_seq"], data["n_static"], cfg["hidden"], cfg["layers"], cfg["dropout"], cfg["head_units"],
        ).to(device)
        predictions = []
        for seed in SEEDS:
            model.load_state_dict(checkpoints[name, seed]["weights"])
            predictions.append(pipeline["forecast_mw"](model, data, "test"))
        index = pipeline["days"][rows].repeat(24) + pd.to_timedelta(np.tile(np.arange(24), len(rows)), unit="h")
        measured = pd.Series(pipeline["L"][rows].ravel(), index=index).reindex(shared["timestamp"])
        np.testing.assert_array_equal(actual, measured.to_numpy())
        ensemble = pd.Series(np.mean(predictions, axis=0).ravel(), index=index)
        forecasts[label] = ensemble.reindex(shared["timestamp"]).to_numpy()

    models = list(forecasts.columns[2:])
    summary = pd.DataFrame([
        {"model": label, "days": len(counts), "hours": len(actual), **metrics(actual, forecasts[label].to_numpy())}
        for label in models
    ])
    hourly = forecasts[models].sub(forecasts["actual_mw"], axis=0).abs()
    hourly.groupby(forecasts["timestamp"].dt.hour).mean().rename_axis("hour").to_csv(OUTPUT / "test_mae_by_hour.csv")
    forecasts.to_csv(OUTPUT / "test_forecasts.csv", index=False)
    summary.to_csv(OUTPUT / "test_summary.csv", index=False)
    return summary

display(test_results(original, configs, chosen, checkpoints, device))


Interpolated single hours: 9
Hours filled from previous week: 24
Grid: 2018-01-01 00:00:00 -> 2026-11-04 23:00:00 (3230 days)
train    2191
val       731
test      308
Name: count, dtype: int64 
Uri days: 7 | unusable target days: [datetime.date(2026, 10, 30)]


,model,days,hours,mae_mw,rmse_mw
0,saved_original,301,7224,1691.652477,2501.626167
1,original_rerun,301,7224,1691.652477,2501.626167


## 6. Check LSTM batch size across three seeds

Vary batch size while preserving the original 14-day lookback, architecture, Nesterov SGD, learning rate, and checkpoint rule. Reuse the three completed batch-32 checkpoints. Score each seed and the average of the three forecasts on 2024?2025 validation data. This follow-up does not score new settings on 2026.


In [6]:
batch_output = OUTPUT / "batch_size"
batch_output.mkdir(exist_ok=True)
batch_sizes = (16, 32, 64)
batch_settings = {
    "configs": {str(batch): {**original, "batch": batch} for batch in batch_sizes},
    "seeds": list(SEEDS),
    "environment_and_inputs": {k: v for k, v in settings.items() if k not in ("configs", "seeds")},
}
batch_settings_path = batch_output / "settings.json"
if batch_settings_path.exists() and json.loads(batch_settings_path.read_text(encoding="utf-8")) != batch_settings:
    raise ValueError("Saved batch-size trials have different settings.")
write_json(batch_settings_path, batch_settings)

batch_data = pipeline["make_data"](original)
batch_rows = sample_rows(batch_data, "val")
batch_actual = pipeline["L"][batch_rows]
batch_records, batch_ensembles = [], []

for batch_size in batch_sizes:
    batch_config = batch_settings["configs"][str(batch_size)]
    predictions = []
    for trial_seed in SEEDS:
        if batch_size == original["batch"]:
            checkpoint = checkpoint_dir / f"original_seed{trial_seed}.pt"
        else:
            checkpoint = batch_output / f"batch{batch_size}_seed{trial_seed}.pt"
        trial = run_trial(pipeline, batch_config, batch_data, trial_seed, checkpoint)
        predictions.append(trial["validation"].numpy())
        batch_records.append({"batch_size": batch_size, "seed": trial_seed, **trial["metrics"]})
        pd.DataFrame(batch_records).to_csv(batch_output / "validation_trials.csv", index=False)
    mean_prediction = np.mean(predictions, axis=0)
    batch_ensembles.append({"batch_size": batch_size, **metrics(batch_actual, mean_prediction)})

batch_trials = pd.DataFrame(batch_records)
batch_summary = batch_trials.groupby("batch_size", as_index=False).agg(
    mean_seed_mae_mw=("mae_mw", "mean"), sd_seed_mae_mw=("mae_mw", "std")
)
batch_summary = batch_summary.merge(pd.DataFrame(batch_ensembles), on="batch_size").rename(
    columns={"mae_mw": "ensemble_mae_mw", "rmse_mw": "ensemble_rmse_mw"}
).sort_values("ensemble_mae_mw")
batch_summary.to_csv(batch_output / "validation_summary.csv", index=False)
display(batch_trials)
display(batch_summary)


batch16_seed42: validation MAE 1644.09 MW
batch16_seed7: validation MAE 1677.34 MW
batch16_seed2026: validation MAE 1684.40 MW
original_seed42: validation MAE 1645.63 MW
original_seed7: validation MAE 1622.42 MW
original_seed2026: validation MAE 1640.39 MW
batch64_seed42: validation MAE 1662.80 MW
batch64_seed7: validation MAE 1649.05 MW
batch64_seed2026: validation MAE 1632.47 MW


,batch_size,seed,mae_mw,rmse_mw,best_epoch,epochs_run,train_seconds
0,16,42,1644.094954,2488.498930,58,83,37.647546
1,16,7,1677.341214,2498.138773,43,68,29.513098
2,16,2026,1684.395681,2506.735193,52,77,31.251914
3,32,42,1645.630515,2493.098041,58,83,24.450034
4,32,7,1622.421099,2479.717252,102,127,36.461424
5,32,2026,1640.386214,2492.095988,70,95,26.880780
6,64,42,1662.800596,2524.349734,118,143,30.998079
7,64,7,1649.046351,2496.806593,114,139,29.869068
8,64,2026,1632.469232,2503.678655,111,136,29.383040


,batch_size,mean_seed_mae_mw,sd_seed_mae_mw,ensemble_mae_mw,ensemble_rmse_mw
1,32,1636.145943,12.17186,1621.871647,2475.195829
2,64,1648.105393,15.18756,1632.568906,2494.760415
0,16,1668.610617,21.52219,1647.828689,2476.507240
